# Exploratory Analysis

This notebook documents the canonical dataset used by the replication package. It verifies the number of cases, inspects derived metadata, and reproduces descriptive summaries used in the paper.

## 1. Load the canonical analysis dataset

The package keeps `Dataset_Construction/processed_data/final_analysis_dataset.csv` unchanged. Repository, PR number, merged, and closed indicators are derived at runtime from `PR_Link` and `Status`.

In [1]:
from pathlib import Path
import sys

# Resolve repository root even when Jupyter starts in a different working directory.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "RQ2_Prompt_Effectiveness_Modeling").exists() and (candidate / "Dataset_Construction").exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from RQ2_Prompt_Effectiveness_Modeling.analysis.common import load_analysis_dataset

df = load_analysis_dataset(ROOT)
df.head()

,Case ID,PR_Link,Conversation_Link,Outcome_Class,Context,Specificity,Verification,Rationale,PQS,PR_Size,...,Case_ID,Repository,PR_Number,Merged,Closed,Generated_Code,Adopted_Code,Resolved,Close_Event,Merge_Event
0,PA-1,https://github.com/Altinn/altinn-broker/pull/259,https://chat.openai.com/share/b7853f70-84b8-47...,PA,1,1,0,Context was scored 1 because the prompt provid...,2,125.0,...,PA-1,Altinn/altinn-broker,259,1,0,1,1,1,0,1
1,PA-2,https://github.com/Hochfrequenz/kohlrahbi/pull...,https://chat.openai.com/share/4ad4c1ad-6f13-4a...,PA,1,1,0,Context was scored 1 because the prompt provid...,2,51.0,...,PA-2,Hochfrequenz/kohlrahbi,158,1,0,1,1,1,0,1
2,PA-3,https://github.com/MartinsOnuoha/what-should-i...,https://chat.openai.com/share/2aa6268a-7a4e-47...,PA,2,2,2,Context was scored 2 because the prompt mentio...,6,300.0,...,PA-3,MartinsOnuoha/what-should-i-design,8,1,0,1,1,1,0,1
3,PA-4,https://github.com/Opetushallitus/ludos/pull/102,https://chat.openai.com/share/bdfcb857-08a3-4f...,PA,1,1,1,Context was scored 1 because the prompt provid...,3,620.0,...,PA-4,Opetushallitus/ludos,102,1,0,1,1,1,0,1
4,PA-5,https://github.com/SharezoneApp/sharezone-app/...,https://chat.openai.com/share/fd82b66d-d949-43...,PA,2,1,1,Context was scored 2 because the prompt mentio...,4,18.0,...,PA-5,SharezoneApp/sharezone-app,980,1,0,1,1,1,0,1


## 2. Dataset shape and outcome-class counts

These counts should align with the final downstream dataset used in the study.

In [2]:
print('Rows:', len(df))
print('Columns:', len(df.columns))
df['Outcome_Class'].value_counts().rename_axis('Outcome_Class').reset_index(name='count')

Rows: 265
Columns: 30


,Outcome_Class,count
0,PA,89
1,NE,80
2,PN,53
3,CL,43


## 3. Prompt quality score distribution

This section summarizes Context, Specificity, Verification, and PQS across the full dataset.

In [3]:
df[['Context','Specificity','Verification','PQS']].describe().round(3)

,Context,Specificity,Verification,PQS
count,265.000,265.000,265.000,265.000
mean,1.475,1.083,0.581,3.140
std,0.622,0.584,0.572,1.359
min,0.000,0.000,0.000,0.000
25%,1.000,1.000,0.000,2.000
50%,2.000,1.000,1.000,3.000
75%,2.000,1.000,1.000,4.000
max,2.000,2.000,2.000,6.000


## 4. PQS by outcome class

This table supports the descriptive comparison across PA, PN, NE, and CL cases.

In [4]:
df.groupby('Outcome_Class')['PQS'].agg(['count','mean','median','std','min','max']).round(3)

,count,mean,median,std,min,max
Outcome_Class,,,,,,
CL,43,3.535,4.0,1.222,0,5
NE,80,2.112,2.0,1.273,0,6
PA,89,3.955,4.0,1.033,2,6
PN,53,3.000,3.0,1.000,1,5


## 5. Pull request size and language summaries

These summaries motivate log-transforming PR size and controlling for project/task characteristics.

In [5]:
display(df['PR_Size'].describe().round(2))
df['PR_Language'].value_counts().head(15).rename_axis('language').reset_index(name='count')

count       261.00
mean       1802.77
std       12171.44
min           0.00
25%          46.00
50%         214.00
75%         754.00
max      182390.00
Name: PR_Size, dtype: float64

,language,count
0,TypeScript,66
1,Python,39
2,Markdown,28
3,YAML,22
4,Go,15
5,JSON,14
6,XML,11
7,Java,9
8,JavaScript,9
9,C++,9
